In [ ]:
# import statements

#import PyPhenom as ppi
import numpy as np
import matplotlib.pyplot as plt

from math import factorial
from scipy.fft import fft2, fftshift
from scipy.signal import fftconvolve
from lmfit import Model, models, minimize, Minimizer, create_params

%matplotlib widget


In [ ]:
# helper functions

def auto_correlation(frame):
    """
    Returns the computed auto_correlation for a given frame.

    Parameter frame: an SEM image
    """

    ft = np.fft.fft2(frame)
    autocorr = np.fft.ifft2(ft * np.conjugate(ft))
    return np.fft.fftshift(autocorr).real


def covariance_matrix(ac):
    """
    Returns the computed covariance matrix given the auto_correlation data.

    Parameter ac: a 2d array with values corresponding to the auto_correlation data at each position i,j
    """

    combine = np.array(np.meshgrid(np.linspace(0, ac.shape[0], ac.shape[0]), # creates 2d array with vector i,j
                                   np.linspace(0, ac.shape[1], ac.shape[1]))).T.reshape(-1, 256, 2)
    
    # covariance equation found from:
    # https://math.stackexchange.com/questions/1939040/get-covariance-of-pixels-in-an-image

    n = 0
    m = 0
    coeff1 = 0
    coeff2 = 0

    for row in range(ac.shape[0]):
        for col in range(ac.shape[1]):
            val = ac[row][col]
            n += val
            m += val ** 2
            coeff1 += val * np.atleast_2d(combine[row][col])

    mean = (1 / n) * coeff1

    for row in range(ac.shape[0]):
        for col in range(ac.shape[1]):
            val = ac[row][col]
            diff = np.atleast_2d(combine[row][col]) - mean
            tpose = diff.T
            coeff2 += val * (diff * tpose)

    cov = (n / (n ** 2 - m)) * coeff2

    return {"covarxx": cov[0][0], "covaryy": cov[1][1], "covarxy": cov[0][1]}


# Define basic Zernike polynomials (on the unit disk)
def zernike_radial(n, m, rho):
    R = np.zeros_like(rho)
    for k in range((n - abs(m)) // 2 + 1):
        coeff = (-1)**k * factorial(n - k)
        coeff /= (factorial(k) *
                  factorial((n + abs(m)) // 2 - k) *
                  factorial((n - abs(m)) // 2 - k))
        R += coeff * rho**(n - 2 * k)
    return R


def zernike(n, m, rho, theta):
    if m > 0:
        return zernike_radial(n, m, rho) * np.cos(m * theta)
    elif m < 0:
        return zernike_radial(n, -m, rho) * np.sin(-m * theta)
    else:
        return zernike_radial(n, 0, rho)


# Mapping Noll index to (n, m)
NOLL_MAP = {
    1:  (0,  0),   # Piston (constant phase offset)
    2:  (1, -1),   # Tilt X (Tip)
    3:  (1,  1),   # Tilt Y (Tilt)
    4:  (2, -2),   # Oblique Astigmatism
    5:  (2,  0),   # Defocus
    6:  (2,  2),   # Vertical Astigmatism
    7:  (3, -3),   # Vertical Trefoil
    8:  (3, -1),   # Vertical Coma
    9:  (3,  1),   # Horizontal Coma
    10: (3,  3),   # Horizontal Trefoil
    11: (4, -4),   # Oblique Tetrafoil
    12: (4, -2),   # Oblique Secondary Astigmatism
    13: (4,  0),   # Primary Spherical Aberration
    14: (4,  2),   # Vertical Secondary Astigmatism
    15: (4,  4),   # Vertical Tetrafoil
}


# SLIGHTLY MODIFIED TO FIT LMFIT MODELS 
def incoherent_zernike_psf(N=256, defocus = None, stig = None): # zernike_coeffs=None):
    if defocus is None: # if zernike_coeffs is None:
        zernike_coeffs = {
            5:2,  # defocus
        }

    else:
        zernike_coeffs = {
            5:defocus
        }

    if stig is not None:
        zernike_coeffs[6] = stig

    x = np.linspace(- 1, 1, N)
    X, Y = np.meshgrid(x, x)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    aperture = rho <= 1 #boolean array

    wavefront = np.zeros_like(rho)

    for j, coeff in zernike_coeffs.items():
        n, m = NOLL_MAP[j]
        Z = zernike(n, m, rho, theta)
        Z[~aperture] = 0
        wavefront += coeff * Z

    # Pupil function with aberration
    phase = 2 * np.pi * wavefront # this could be scaled by the wavelength.
    # suggest to not to do this in this case that adds to the confusion ==> this is not the same as a coherent point source!

    pupil = np.exp(1j * phase) 
    pupil[~aperture] = 0

    # Coherent PSF
    h = fftshift(fft2(pupil)) #The point spread function (PSF), which is given by the square of the absolute value of the 
    psf_coh = np.abs(h)**2    #Fourier transform of the wavefront, shows what the intensity image of a point source would look like.

    # Incoherent PSF: autocorrelation of coherent intensity
    psf_incoh = fftconvolve(psf_coh, psf_coh, mode='same')
    psf_incoh /= psf_incoh.sum()

    return psf_incoh


def incoh_psf_fit(ac, df, st):
    model = Model(incoherent_zernike_psf)#, independent_vars=['x', 'y']) 

    nrows = ac.shape[0]
    ncols = ac.shape[1]

    x_val, y_val = np.meshgrid(np.linspace(0, ncols, ncols), np.linspace(0, nrows, nrows))

    pars = create_params(N = dict(value = 256, vary = False),
                         defocus = dict(value = df, vary = True),
                         stig = dict(value = st, vary = True))

    result = model.fit(ac, x = x_val, y = y_val, params = pars) # uses the above parameters and  model to fit the data

    plt.figure()
    plt.imshow(acdata)
    plt.contour(result.best_fit)

    return result


def obj_func(pars):
    """
    Returns the covariance matrix for a given set of pars.

    Parameter phenom: a Phenom object
    Parameter pars: A lmfit Parameters object
    """

    p = pars.valuesdict()
    phenom.SetSemWD(p['wd'])
    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    #phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    acdata = auto_correlation(img_array) # calculates the auto_correlation data
    # acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata.[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2 - 1]
    return (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1]) * (-1)

    return acdata.sum(axis=(0,1)) * (-1)

    # fitted_vals = incoh_psf_fit(acdata, -2, .3).best_values # gets the coeffs for the fitted equation

    # calculates the covar matrix based off of the fitted data
    # cov = covariance_matrix(incoherent_zernike_psf(256, fitted_vals["defocus"], fitted_vals["stig"])) 

    # return np.sqrt(cov["covarxx"]**2 + cov["covaryy"]**2)


def focus_2(wd_val_min, wd_val_max):
    """
    Sets the working distance, x-direction stigmate, and y-direction stigmate to the values needed for a focused image.

    Parameter phenom: a Phenom object
    Parameter num_pic: the number of pictures taken
    Parameter val_min: the minimum value of wd, xst, or yst that a picture is taken at
    Parameter val_max: the maximum value of wd, xst, or yst that a picture is taken at
    """
    
    params = create_params(wd = dict(value = wd_val_min, vary = True, min = wd_val_min, max = wd_val_max, brute_step = 0.00005)) # change brute step

    fitter = Minimizer(obj_func, params)
    result = fitter.minimize(method = 'brute')

    print(result.brute_x0)
    print(result.brute_fval)

    plt.figure()
    plt.plot(result.brute_grid, result.brute_Jout)

    #phenom.SetSemWD(result.ampgo_x0)
    phenom.SetSemWD(result.brute_x0)

In [ ]:
# main thing 

# connect to SEM 
PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

phenom.SemAutoContrastBrightness()

wd_val_min = 0.0090 #0.008 #0.002
wd_val_max = 0.01 #0.011 #0.025 #0.12198 is max value
xst_val_min = -1
xst_val_max = 1
yst_val_min = -1
yst_val_max = 1

focus_2(wd_val_min, wd_val_max)
#focus_3(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)

#focus(phenom, "wd", num_pic, wd_val_min, wd_val_max)
#auto_focus(phenom, num_pic, wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)


In [ ]:
xdim = 256 # x and y dimensions of image taken
ydim = 256
stack = 8
HDR = False
scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

phenom.SemAutoContrastBrightness()
acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

img_array = np.array(acq.image) # converts image into an array
acdata = auto_correlation(img_array) # calculates the auto_correlation data
cov = covariance_matrix(acdata)

print(acdata)
plt.figure()
plt.imshow(acdata)
plt.contour(cov.best_fit)
print(covariance_matrix(acdata))

#phenom.SemAutoContrastBrightness()
acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

img_array = np.array(acq.image) # converts image into an array
acdata = auto_correlation(img_array) # calculates the auto_correlation data
acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata.mean()
cov = covariance_matrix(acdata)

print(acdata)
plt.figure()
plt.imshow(acdata)
plt.contour(cov.best_fit)
print(covariance_matrix(acdata))